# [Hands-On] Self-Attention Mechanism in GPT-OSS 20B

- Author: Sangkeun Jung (hugmanskj@gmail.com)

> Educational Purpose

**Copyright**: All rights reserved

---

## Overview

In this hands-on tutorial, we will dissect and implement the core of transformer models:
the **Self-Attention mechanism**. We will build a complete attention layer from scratch
and verify it produces identical results to HuggingFace's implementation.

We will:

- Understand Query, Key, Value projections
- Implement Scaled Dot-Product Attention
- Learn Grouped Query Attention (GQA) for efficiency
- Implement Attention Sink tokens (GPT-OSS specific)
- Apply RoPE to Query and Key vectors
- Visualize attention patterns
- Verify exact match with reference implementation

This is the heart of the transformer - where tokens learn to attend to each other!

## 1. Theoretical Foundation

### The Attention Mechanism

**Intuition**: When reading "The animal didn't cross the street because **it** was too tired",
how do we know "it" refers to "animal"? We attend to relevant context!

**Library Analogy**:
```
Query:  "I need books about machine learning"
Keys:   [Math, ML, Cooking, History, ...]
Values: [Math content, ML content, Cooking content, ...]

Process:
1. Compare Query with each Key → similarity scores
2. Convert scores to probabilities (softmax)
3. Weighted average of Values
Result: Mostly ML content!
```

### Mathematical Formulation

**Scaled Dot-Product Attention**:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

Where:
- $Q$ (Query): What I'm looking for - shape $[batch, heads, seq, d_k]$
- $K$ (Key): What I have - shape $[batch, heads, seq, d_k]$
- $V$ (Value): Actual content - shape $[batch, heads, seq, d_v]$
- $d_k$: Key/Query dimension (for scaling)

**Why scale by** $\sqrt{d_k}$?
- Dot products grow with dimension
- Large values → saturated softmax → vanishing gradients
- Scaling keeps values in reasonable range

### Multi-Head Attention

Instead of single attention, use multiple "heads" in parallel:

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O
$$

$$
\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$

**Benefits**:
- Different heads learn different patterns
- Head 1: Syntactic relationships (subject-verb)
- Head 2: Semantic similarity
- Head 3: Coreference resolution
- etc.

### Grouped Query Attention (GQA)

**Problem**: Standard Multi-Head Attention uses too much memory (KV cache)

**Solution**: Share Key and Value across multiple Query heads!

```
Standard MHA:  64 Q heads, 64 K heads, 64 V heads
GQA:           64 Q heads,  8 K heads,  8 V heads
               → Each K/V serves 8 Q heads
```

**Memory Savings**: 8x smaller KV cache!

### Attention Sinks (GPT-OSS Specific)

**Innovation**: Add an extra "sink" column to attention scores.

**Purpose**:
- When no token is relevant, attend to sink instead
- Prevents forced uniform attention
- Learned parameter per head

**Implementation**:
```
scores = QK^T                    # [batch, heads, seq, seq]
sinks = learned_params          # [heads]
combined = concat(scores, sinks) # [batch, heads, seq, seq+1]
probs = softmax(combined)
probs = probs[..., :-1]          # Remove sink column
output = probs @ V
```

---

## 2. Import Libraries and Helper Functions

In [1]:
import sys
import math
from typing import Optional, Tuple
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub.errors import HfHubHTTPError, RepositoryNotFoundError


def set_seed(seed: int = 42):
    """Set random seed for reproducibility"""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def format_tensor_slice(tensor: torch.Tensor, limit: int = 8) -> str:
    """Format tensor values for readable display"""
    flat = tensor.flatten()
    elems = flat[:limit].tolist()
    pieces = ", ".join(f"{x:.4f}" for x in elems)
    return f"{pieces}, ..." if flat.numel() > limit else pieces


def load_with_token(load_fn, token: str | None, **kwargs):
    """Load model/tokenizer with authentication token"""
    auth_kwargs = {"token": token} if token else {}
    try:
        return load_fn(**auth_kwargs, **kwargs)
    except TypeError:
        auth_kwargs = {"use_auth_token": token} if token else {}
        return load_fn(**auth_kwargs, **kwargs)


def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    Repeat key/value tensors for Grouped Query Attention.

    This is equivalent to torch.repeat_interleave(x, dim=1, repeats=n_rep)
    but optimized for this specific use case.

    Args:
        hidden_states: [batch, num_kv_heads, seq_len, head_dim]
        n_rep: Number of times to repeat

    Returns:
        [batch, num_kv_heads * n_rep, seq_len, head_dim]
    """
    batch, num_kv_heads, seq_len, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states

    hidden_states = hidden_states[:, :, None, :, :].expand(
        batch, num_kv_heads, n_rep, seq_len, head_dim
    )
    return hidden_states.reshape(batch, num_kv_heads * n_rep, seq_len, head_dim)


print("Libraries and helper functions loaded successfully!")

Libraries and helper functions loaded successfully!


---

## 3. Configuration and Settings

In [2]:
# ============================================================
# Configuration
# ============================================================

# Model settings
MODEL_NAME = "openai/gpt-oss-20b"
DTYPE_STR = "bfloat16"
DEVICE_MAP = "auto"
HF_TOKEN = None

# Test input
PROMPT = "hello gpt"

# Seed for reproducibility
SEED = 42

# Device selection
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU (Warning: This will be slow)")

# Data type mapping
DTYPE_MAP = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}

if DTYPE_STR not in DTYPE_MAP:
    raise ValueError(f"Unknown dtype: {DTYPE_STR}. Available: {list(DTYPE_MAP.keys())}")

TORCH_DTYPE = DTYPE_MAP[DTYPE_STR]

# Set seed
set_seed(SEED)

print(f"Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Prompt: '{PROMPT}'")
print(f"  Data type: {TORCH_DTYPE}")
print(f"  Device: {DEVICE}")
print(f"  Seed: {SEED}")

Using CPU (Warning: This will be slow)
Configuration:
  Model: openai/gpt-oss-20b
  Prompt: 'hello gpt'
  Data type: torch.bfloat16
  Device: cpu
  Seed: 42


---

## 4. Load Reference Model and Extract Components

In [3]:
print("Loading HuggingFace GPT-OSS 20B model...")
print("-" * 60)

try:
    # Load tokenizer
    print("Loading tokenizer...")
    tokenizer = load_with_token(
        AutoTokenizer.from_pretrained,
        HF_TOKEN,
        pretrained_model_name_or_path=MODEL_NAME,
    )
    print("Tokenizer loaded successfully!")

    # Load model
    print("Loading model...")
    reference_model = load_with_token(
        AutoModelForCausalLM.from_pretrained,
        HF_TOKEN,
        pretrained_model_name_or_path=MODEL_NAME,
        torch_dtype=TORCH_DTYPE,
        device_map=DEVICE_MAP,
    )
    print("Model loaded successfully!")

except (RepositoryNotFoundError, HfHubHTTPError) as err:
    print(f"Failed to load model: {err}", file=sys.stderr)
    raise
except OSError as err:
    print(f"Failed to load model: {err}", file=sys.stderr)
    raise

# Extract configuration
config = reference_model.config

print("\n" + "=" * 60)
print("Model Configuration (Attention-related)")
print("=" * 60)
print(f"Hidden size: {config.hidden_size}")
print(f"Num attention heads: {config.num_attention_heads}")
print(f"Num key-value heads: {config.num_key_value_heads}")
print(f"Head dimension: {getattr(config, 'head_dim', config.hidden_size // config.num_attention_heads)}")
print(f"GQA ratio: {config.num_attention_heads // config.num_key_value_heads}:1")
print(f"  → Each K/V head serves {config.num_attention_heads // config.num_key_value_heads} Q heads")
print("=" * 60)

Loading HuggingFace GPT-OSS 20B model...
------------------------------------------------------------
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Model loaded successfully!

Model Configuration (Attention-related)
Hidden size: 2880
Num attention heads: 64
Num key-value heads: 8
Head dimension: 64
GQA ratio: 8:1
  → Each K/V head serves 8 Q heads


---

## 5. Tokenize and Get Input Embeddings

In [4]:
print(f"Input text: '{PROMPT}'")
print("-" * 60)

# Tokenize
encoded = tokenizer(
    PROMPT,
    return_tensors="pt",
    add_special_tokens=False,
)

input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

# Display tokenization results
print("Tokenization Results:")
print(f"Number of tokens: {len(tokens)}")
print(f"Tokens: {[t.replace('Ġ', '▁') for t in tokens]}")
print(f"Token IDs: {input_ids.tolist()}")

Input text: 'hello gpt'
------------------------------------------------------------
Tokenization Results:
Number of tokens: 3
Tokens: ['hello', '▁g', 'pt']
Token IDs: [[24912, 329, 555]]


---

## 5. Reusing Components from Previous Tutorials

We build incrementally! Below are implementations from previous tutorials:
- **CustomTokenEmbedding**: from `01_[hands-on]_token_embedding.py`
- **CustomRotaryEmbedding**: from `02_[hands-on]_rope_and_attention_mask.py`
- **CustomCausalMask**: from `02_[hands-on]_rope_and_attention_mask.py`

In [5]:
# ============================================================
# REUSED FROM TUTORIAL 01: CustomTokenEmbedding
# ============================================================
# Originally implemented in: 01_[hands-on]_token_embedding.py
# ============================================================

class CustomTokenEmbedding(nn.Module):
    """Custom implementation of token embedding layer."""

    def __init__(
        self,
        vocab_size: int,
        hidden_size: int,
        dtype: torch.dtype = torch.float32,
        device: torch.device | None = None,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.weight = nn.Parameter(
            torch.empty(vocab_size, hidden_size, dtype=dtype, device=device)
        )
        with torch.no_grad():
            nn.init.normal_(self.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        return torch.embedding(self.weight, input_ids)

    @classmethod
    def from_pretrained(cls, reference_embedding_layer):
        vocab_size, hidden_size = reference_embedding_layer.weight.shape
        dtype = reference_embedding_layer.weight.dtype
        device = reference_embedding_layer.weight.device
        custom_embedding = cls(vocab_size, hidden_size, dtype, device)
        with torch.no_grad():
            custom_embedding.weight.copy_(reference_embedding_layer.weight)
        return custom_embedding

In [6]:
# ============================================================
# REUSED FROM TUTORIAL 02: CustomRotaryEmbedding
# ============================================================
# Originally implemented in: 02_[hands-on]_rope_and_attention_mask.py
# ============================================================

class CustomRotaryEmbedding(nn.Module):
    """Custom implementation of Rotary Position Embedding with YaRN scaling."""

    def __init__(
        self,
        head_dim: int,
        base: float = 10000.0,
        dtype: torch.dtype = torch.float32,
        initial_context_length: int = 4096,
        scaling_factor: float = 1.0,
        ntk_alpha: float = 1.0,
        ntk_beta: float = 32.0,
        device: torch.device | None = None,
    ):
        super().__init__()
        self.head_dim = head_dim
        self.base = base
        self.dtype = dtype
        self.initial_context_length = initial_context_length
        self.scaling_factor = scaling_factor
        self.ntk_alpha = ntk_alpha
        self.ntk_beta = ntk_beta
        self.device = device

    def _compute_concentration_and_inv_freq(self):
        freq = self.base ** (
            torch.arange(0, self.head_dim, 2, dtype=torch.float32, device=self.device)
            / self.head_dim
        )

        if self.scaling_factor > 1.0:
            concentration = 0.1 * math.log(self.scaling_factor) + 1.0
            d_half = self.head_dim / 2
            low = (
                d_half
                * math.log(self.initial_context_length / (self.ntk_beta * 2 * math.pi))
                / math.log(self.base)
            )
            high = (
                d_half
                * math.log(self.initial_context_length / (self.ntk_alpha * 2 * math.pi))
                / math.log(self.base)
            )
            interpolation = 1.0 / (self.scaling_factor * freq)
            extrapolation = 1.0 / freq
            ramp = (
                torch.arange(d_half, dtype=torch.float32, device=freq.device) - low
            ) / (high - low)
            mask = 1 - ramp.clamp(0, 1)
            inv_freq = interpolation * (1 - mask) + extrapolation * mask
        else:
            concentration = 1.0
            inv_freq = 1.0 / freq

        return concentration, inv_freq

    def forward(self, value_states, position_ids):
        """Generate RoPE cos/sin for given positions."""
        seq_len = position_ids.shape[-1]
        concentration, inv_freq = self._compute_concentration_and_inv_freq()
        positions = torch.arange(seq_len, dtype=torch.float32, device=self.device)
        freqs = torch.einsum("i,j->ij", positions, inv_freq)
        cos = torch.cos(freqs) * concentration
        sin = torch.sin(freqs) * concentration
        return cos.unsqueeze(0).to(self.dtype), sin.unsqueeze(0).to(self.dtype)

In [7]:
# ============================================================
# REUSED FROM TUTORIAL 02: CustomCausalMask
# ============================================================
# Originally implemented in: 02_[hands-on]_rope_and_attention_mask.py
# ============================================================

class CustomCausalMask:
    """Custom implementation of causal attention mask."""

    @staticmethod
    def create_causal_mask(
        seq_len: int,
        sliding_window: int | None = None,
        device: torch.device | None = None,
        dtype: torch.dtype = torch.float32,
    ) -> torch.Tensor:
        """Create causal mask that prevents attending to future tokens."""
        mask = torch.zeros(seq_len, seq_len, dtype=dtype, device=device)
        future_mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        mask = mask.masked_fill(future_mask, float("-inf"))

        if sliding_window is not None:
            past_mask = torch.tril(
                torch.ones(seq_len, seq_len, device=device), diagonal=-sliding_window - 1
            ).bool()
            mask = mask.masked_fill(past_mask, float("-inf"))

        return mask


print("✓ Components reused from previous tutorials")

✓ Components reused from previous tutorials


### Create Custom Instances

Now let's instantiate our custom components using the reference model's weights.

In [8]:
print("Creating custom component instances...")

# Create custom embedding
reference_embedding = reference_model.get_input_embeddings()
custom_embedding = CustomTokenEmbedding.from_pretrained(reference_embedding)
print(f"✓ CustomTokenEmbedding: {custom_embedding.weight.shape}")

# Get embedding device for later use
embedding_device = custom_embedding.weight.device

# Create custom RoPE
rope_config = config.rope_scaling
custom_rope = CustomRotaryEmbedding(
    head_dim=config.head_dim,
    base=config.rope_theta,
    dtype=TORCH_DTYPE,
    initial_context_length=rope_config.get('original_max_position_embeddings', 4096),
    scaling_factor=rope_config.get('factor', 32.0),
    ntk_alpha=rope_config.get('low_freq_factor', 1.0),
    ntk_beta=rope_config.get('high_freq_factor', 32.0),
    device=embedding_device,
)
print(f"✓ CustomRotaryEmbedding: head_dim={custom_rope.head_dim}, scaling={custom_rope.scaling_factor}")

print("Custom components ready!")

Creating custom component instances...
✓ CustomTokenEmbedding: torch.Size([201088, 2880])
✓ CustomRotaryEmbedding: head_dim=64, scaling=32.0
Custom components ready!


### Use Custom Embedding

Compute embeddings using our custom implementation instead of HuggingFace.

In [9]:
print("Computing embeddings with custom implementation...")

# Move input_ids to correct device
input_ids = input_ids.to(embedding_device)

# Use custom embedding
with torch.no_grad():
    hidden_states = custom_embedding(input_ids)

print(f"✓ Hidden states computed using CustomTokenEmbedding")
print(f"  Shape: {tuple(hidden_states.shape)}")

Computing embeddings with custom implementation...
✓ Hidden states computed using CustomTokenEmbedding
  Shape: (1, 3, 2880)


---

## 6. Create Attention Mask and Position IDs

In [10]:
print("Creating attention mask and position information...")

# Position IDs: [0, 1, 2, ...]
position_ids = torch.arange(
    input_ids.shape[-1],
    device=embedding_device,
    dtype=torch.long
).unsqueeze(0)

# Cache position for mask generation
cache_position = torch.arange(
    input_ids.shape[-1],
    device=embedding_device,
    dtype=torch.long
)

# Create causal mask using our custom implementation
seq_len = input_ids.shape[-1]
causal_mask = CustomCausalMask.create_causal_mask(
    seq_len=seq_len,
    sliding_window=None,  # GPT-OSS uses full causal mask (no sliding window)
    device=embedding_device,
    dtype=TORCH_DTYPE,
)

# Add batch and head dimensions: [seq, seq] → [1, 1, seq, seq]
causal_mask = causal_mask[None, None, :, :]

print(f"Position IDs: {position_ids.tolist()}")
print(f"Causal mask shape: {tuple(causal_mask.shape)}")
print(f"Causal mask (2D view):")
print(causal_mask[0, 0])

Creating attention mask and position information...
Position IDs: [[0, 1, 2]]
Causal mask shape: (1, 1, 3, 3)
Causal mask (2D view):
tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]], dtype=torch.bfloat16)


---

## 7. Custom Self-Attention Implementation

### Architecture Overview

```
Input: [batch, seq_len, hidden_size]
  ↓
Q/K/V Projections (Linear layers)
  ↓
Reshape to multi-head: [batch, heads, seq_len, head_dim]
  ↓
GQA: Repeat K/V to match Q heads
  ↓
Apply RoPE to Q and K
  ↓
Scaled Dot-Product Attention:
  - scores = Q @ K^T * scaling
  - apply causal mask
  - add sink tokens
  - softmax
  - remove sink
  - context = probs @ V
  ↓
Concat heads and project output
  ↓
Output: [batch, seq_len, hidden_size]
```

In [18]:
@dataclass
class AttentionConfig:
    """Configuration for attention module"""
    hidden_size: int
    num_attention_heads: int
    num_key_value_heads: int
    head_dim: int
    rope_theta: float
    rope_scaling: dict


class CustomSelfAttention(nn.Module):
    """
    Custom implementation of GPT-OSS Self-Attention with GQA and Sink tokens.

    This implementation exactly mirrors HuggingFace's GptOssAttention for
    educational purposes, with explicit step-by-step operations.
    """

    def __init__(
        self,
        config: AttentionConfig,
        layer_idx: int = 0,
        dtype: torch.dtype = torch.bfloat16,
        device: torch.device = None,
        rotary_emb=None,
    ):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_kv_groups = self.num_heads // self.num_kv_heads

        # Attention scaling factor
        self.scaling = 1.0 / math.sqrt(self.head_dim)

        # RoPE (passed from model level)
        self.rotary_emb = rotary_emb

        # ========================================
        # Linear Projection Layers
        # ========================================

        # Query projection: [hidden_size] → [num_heads * head_dim]
        self.q_proj = nn.Linear(
            self.hidden_size,
            self.num_heads * self.head_dim,
            bias=True,
            dtype=dtype,
            device=device,
        )

        # Key projection: [hidden_size] → [num_kv_heads * head_dim]
        self.k_proj = nn.Linear(
            self.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=True,
            dtype=dtype,
            device=device,
        )

        # Value projection: [hidden_size] → [num_kv_heads * head_dim]
        self.v_proj = nn.Linear(
            self.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=True,
            dtype=dtype,
            device=device,
        )

        # Output projection: [num_heads * head_dim] → [hidden_size]
        self.o_proj = nn.Linear(
            self.num_heads * self.head_dim,
            self.hidden_size,
            bias=True,
            dtype=dtype,
            device=device,
        )

        # ========================================
        # Sink Tokens (Learnable parameters)
        # ========================================
        self.sinks = nn.Parameter(
            torch.zeros(self.num_heads, dtype=dtype, device=device)
        )


    @classmethod
    def from_pretrained_layer(cls, reference_layer, custom_rotary_emb, layer_idx: int = 0):
        """
        Create custom attention by copying weights from HuggingFace layer.

        Args:
            reference_layer: HuggingFace GptOssAttention layer
            custom_rotary_emb: Our CustomRotaryEmbedding instance
            layer_idx: Layer index in the model

        Returns:
            CustomSelfAttention instance with copied weights
        """
        # Extract config
        hf_config = reference_layer.config
        attn_config = AttentionConfig(
            hidden_size=hf_config.hidden_size,
            num_attention_heads=hf_config.num_attention_heads,
            num_key_value_heads=hf_config.num_key_value_heads,
            head_dim=reference_layer.head_dim,
            rope_theta=hf_config.rope_theta,
            rope_scaling=hf_config.rope_scaling,
        )

        # Use our custom RoPE implementation
        rotary_emb = custom_rotary_emb

        # Create instance
        custom_attn = cls(
            config=attn_config,
            layer_idx=layer_idx,
            dtype=reference_layer.q_proj.weight.dtype,
            device=reference_layer.q_proj.weight.device,
            rotary_emb=rotary_emb,
        )

        # Copy weights
        with torch.no_grad():
            custom_attn.q_proj.weight.copy_(reference_layer.q_proj.weight)
            custom_attn.q_proj.bias.copy_(reference_layer.q_proj.bias)
            custom_attn.k_proj.weight.copy_(reference_layer.k_proj.weight)
            custom_attn.k_proj.bias.copy_(reference_layer.k_proj.bias)
            custom_attn.v_proj.weight.copy_(reference_layer.v_proj.weight)
            custom_attn.v_proj.bias.copy_(reference_layer.v_proj.bias)
            custom_attn.o_proj.weight.copy_(reference_layer.o_proj.weight)
            custom_attn.o_proj.bias.copy_(reference_layer.o_proj.bias)
            custom_attn.sinks.copy_(reference_layer.sinks)

        return custom_attn

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Forward pass of self-attention.

        Args:
            hidden_states: Input tensor [batch, seq_len, hidden_size]
            attention_mask: Causal mask [batch, 1, seq_len, seq_len]
            position_ids: Position indices [batch, seq_len]

        Returns:
            Output tensor [batch, seq_len, hidden_size]
        """
        batch_size, seq_len, _ = hidden_states.shape

        # ========================================
        # Step 1: Linear Projections
        # ========================================
        # Project input to Q, K, V
        query_states = self.q_proj(hidden_states)
        key_states = self.k_proj(hidden_states)
        value_states = self.v_proj(hidden_states)


        # ========================================
        # Step 2: Reshape to Multi-Head Format
        # ========================================

        # Q: [batch, seq, num_heads * head_dim] → [batch, num_heads, seq, head_dim]
        query_states = query_states.view(
        batch_size, seq_len, self.num_heads, self.head_dim
        ).transpose(1, 2)
        
        # K: [batch, seq, num_kv_heads * head_dim] → [batch, num_kv_heads, seq, head_dim]
        key_states = key_states.view(
            batch_size, seq_len, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        # V: [batch, seq, num_kv_heads * head_dim] → [batch, num_kv_heads, seq, head_dim]
        value_states = value_states.view(
            batch_size, seq_len, self.num_kv_heads, self.head_dim
        ).transpose(1, 2)

        # ========================================
        # Step 3: Apply RoPE to Q and K
        # ========================================
        # RoPE (Rotary Position Embedding) encodes position information
        # through rotation in complex space.
        #
        # GPT-OSS uses **Blocked Pairing** method:
        # - Dimension layout: [d0, d1, ..., d31, d32, d33, ..., d63] for head_dim=64
        # - Pairing: (d0,d32), (d1,d33), ..., (d31,d63)
        # - First half pairs with second half
        #
        # This is different from Interleaved Pairing:
        # - Would pair: (d0,d1), (d2,d3), ..., (d62,d63)
        # - Adjacent dimensions paired together
        #
        # Both methods are mathematically equivalent for position encoding,
        # but blocked pairing is simpler to implement with chunk(2, dim=-1)
        cos, sin = self.rotary_emb(value_states, position_ids=position_ids)
        query_states, key_states = self.apply_rotary_pos_emb(
            query_states, key_states, cos, sin
        )

        # ========================================
        # Step 4: GQA - Repeat K and V
        # ========================================
        # Repeat K/V to match number of Q heads
        key_states = repeat_kv(key_states, self.num_kv_groups)
        value_states = repeat_kv(value_states, self.num_kv_groups)

        # Now all have shape: [batch, num_heads, seq, head_dim]

        # ========================================
        # Step 5: Scaled Dot-Product Attention
        # ========================================

        # Compute attention scores: Q @ K^T
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3))

        # Scale
        attn_weights = attn_weights * self.scaling

        # Apply causal mask
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask.to(attn_weights.dtype)

        # ========================================
        # Step 6: Add Sink Tokens
        # ========================================
        # Expand sinks to match batch and sequence dimensions
        # [num_heads] → [batch, num_heads, seq, 1]
        sinks = self.sinks.view(1, -1, 1, 1).expand(batch_size, -1, seq_len, -1)
        
        # Concatenate with attention scores
        attn_weights = torch.cat([attn_weights, sinks.to(attn_weights.dtype)], dim=-1)

        # [batch, heads, seq, seq] + [batch, heads, seq, 1] → [batch, heads, seq, seq+1]

        # ========================================
        # Step 7: Softmax
        # ========================================
        # Numerical stability: subtract max before softmax\
        attn_weights = attn_weights - attn_weights.max(dim=-1, keepdim=True).values
        attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)

        # Remove sink column
        attn_weights = attn_weights[..., :-1]

        # ========================================
        # Step 8: Compute Context
        # ========================================
        # attn_weights @ V: [batch, heads, seq, seq] @ [batch, heads, seq, head_dim]
        attn_output = torch.matmul(attn_weights, value_states)

        # ========================================
        # Step 9: Concatenate Heads
        # ========================================
        # [batch, heads, seq, head_dim] → [batch, seq, heads, head_dim]
        attn_output = attn_output.transpose(1, 2).contiguous()

        # [batch, seq, heads, head_dim] → [batch, seq, heads * head_dim]
        attn_output = attn_output.reshape(batch_size, seq_len, self.num_heads * self.head_dim)

        # ========================================
        # Step 10: Output Projection
        # ========================================
        attn_output = self.o_proj(attn_output)

        return attn_output

    def apply_rotary_pos_emb(self, q, k, cos, sin):
        """
        Apply RoPE to query and key tensors using **Blocked Pairing**.

        RoPE Pairing Methods:
        ---------------------
        There are two ways to pair dimensions for rotation:

        1. **Interleaved Pairing**: (x0, x1), (x2, x3), (x4, x5), ...
           - Pairs adjacent dimensions: [d0, d1, d2, d3, ...] → [(d0,d1), (d2,d3), ...]
           - Used in some implementations (e.g., original RoFormer)

        2. **Blocked Pairing**: (x0, x_{d/2}), (x1, x_{d/2+1}), ...
           - Splits into two halves: [d0...d_{n-1}, d_n...d_{2n-1}] → [(d0,d_n), (d1,d_{n+1}), ...]
           - **GPT-OSS uses this method!**

        Why Blocked Pairing?
        --------------------
        - Easier implementation: chunk(2, dim=-1) naturally splits into two blocks
        - Better for hardware: contiguous memory access patterns
        - Equivalent mathematical properties to interleaved pairing

        Implementation:
        ---------------
        For blocked pairing, rotate_half() splits [d0...d_{n-1}, d_n...d_{2n-1}]
        into x1=[d0...d_{n-1}] and x2=[d_n...d_{2n-1}], then returns [-x2, x1]

        Args:
            q, k: shape [batch, num_heads, seq_len, head_dim]
            cos, sin: shape [batch, seq_len, head_dim//2]

        Returns:
            q_embed, k_embed: Rotated query and key tensors
        """
        # cos/sin shape: [batch, seq_len, head_dim//2]
        # q/k shape: [batch, num_heads, seq_len, head_dim]

        # Unsqueeze to add head dimension: [batch, 1, seq_len, head_dim//2]
        cos = cos.unsqueeze(1)
        sin = sin.unsqueeze(1)

        # Repeat for full head_dim: [batch, 1, seq_len, head_dim]
        # This duplicates cos/sin to match the blocked pairing structure
        cos = torch.cat([cos, cos], dim=-1)
        sin = torch.cat([sin, sin], dim=-1)

        # Apply rotation using blocked pairing
        def rotate_half(x):
            """
            Rotate half the dimensions (blocked pairing method).

            For blocked pairing:
            - x.chunk(2, dim=-1) splits [d0...d_{n-1}, d_n...d_{2n-1}]
            - Returns [-x2, x1] where x1=[d0...d_{n-1}], x2=[d_n...d_{2n-1}]

            This implements the rotation in complex plane:
            - Real part: x1, Imaginary part: x2
            - After rotation: Real=-x2, Imaginary=x1
            """
            x1, x2 = x.chunk(2, dim=-1)  # Split into two blocks
            return torch.cat((-x2, x1), dim=-1)  # Rotate: [-Im, Re]

        # Apply RoPE: x * cos + rotate_half(x) * sin
        # This is equivalent to complex multiplication: x * e^(i*theta)
        q_embed = (q * cos) + (rotate_half(q) * sin)
        k_embed = (k * cos) + (rotate_half(k) * sin)

        return q_embed, k_embed


print("Custom Self-Attention implementation complete!")

Custom Self-Attention implementation complete!


---

## 8. Create Custom Attention Instance

In [ ]:
print("Creating custom attention layer...")

# Get first attention layer from reference model
reference_attention = reference_model.model.layers[0].self_attn

# Create custom attention with copied weights and our custom RoPE
custom_attention = CustomSelfAttention.from_pretrained_layer(
    reference_attention,
    custom_rope,  # Use our CustomRotaryEmbedding
    layer_idx=0
)

print(f"Custom attention created!")
print(f"  Num Q heads: {custom_attention.num_heads}")
print(f"  Num KV heads: {custom_attention.num_kv_heads}")
print(f"  GQA ratio: {custom_attention.num_kv_groups}:1")
print(f"  Head dim: {custom_attention.head_dim}")
print(f"  Scaling factor: {custom_attention.scaling:.6f}")

---

## 9. Run Forward Pass and Verify

In [ ]:
print("Running forward pass...")
print("=" * 60)

# Custom forward pass
print("1. Running custom attention...")
with torch.no_grad():
    custom_output = custom_attention(
        hidden_states=hidden_states,
        attention_mask=causal_mask,
        position_ids=position_ids,
    )

print(f"   Output shape: {tuple(custom_output.shape)}")

# Reference forward pass
print("\n2. Running reference attention...")
with torch.no_grad():
    # Get position embeddings
    cos, sin = reference_model.model.rotary_emb(hidden_states, position_ids=position_ids)
    position_embeddings = (cos, sin)

    reference_output = reference_attention(
        hidden_states=hidden_states,
        attention_mask=causal_mask,
        position_ids=position_ids,
        position_embeddings=position_embeddings,
    )[0]  # Get only the output tensor

print(f"   Output shape: {tuple(reference_output.shape)}")

# Compare outputs
print("\n3. Computing differences...")
abs_diff = (custom_output - reference_output).abs()
max_diff = abs_diff.max().item()
mean_diff = abs_diff.mean().item()

print(f"   Maximum absolute difference: {max_diff:.6e}")
print(f"   Mean absolute difference:    {mean_diff:.6e}")

# Verification
tolerance = 1e-5 if custom_output.dtype == torch.float32 else 1e-3

if max_diff < tolerance:
    print(f"\n✓ ATTENTION VERIFICATION PASSED!")
    print(f"  Maximum difference ({max_diff:.6e}) is within tolerance ({tolerance})")
    print(f"  Custom implementation produces identical results!")
else:
    print(f"\n✗ ATTENTION VERIFICATION FAILED!")
    print(f"  Maximum difference ({max_diff:.6e}) exceeds tolerance ({tolerance})")

print("=" * 60)

---

## 10. Visualization: Attention Patterns

Let's visualize what the attention mechanism is actually doing!

In [ ]:
print("Generating attention pattern visualization...")

# We need to extract attention weights
# Re-run forward pass with explicit weight extraction
batch_size, seq_len, _ = hidden_states.shape

with torch.no_grad():
    # Project to Q, K, V
    query_states = custom_attention.q_proj(hidden_states)
    key_states = custom_attention.k_proj(hidden_states)
    value_states = custom_attention.v_proj(hidden_states)

    # Reshape
    query_states = query_states.view(
        batch_size, seq_len, custom_attention.num_heads, custom_attention.head_dim
    ).transpose(1, 2)

    key_states = key_states.view(
        batch_size, seq_len, custom_attention.num_kv_heads, custom_attention.head_dim
    ).transpose(1, 2)

    value_states = value_states.view(
        batch_size, seq_len, custom_attention.num_kv_heads, custom_attention.head_dim
    ).transpose(1, 2)

    # Apply RoPE
    cos, sin = custom_attention.rotary_emb(value_states, position_ids=position_ids)
    query_states, key_states = custom_attention.apply_rotary_pos_emb(
        query_states, key_states, cos, sin
    )

    # Repeat K/V for GQA
    key_states = repeat_kv(key_states, custom_attention.num_kv_groups)
    value_states = repeat_kv(value_states, custom_attention.num_kv_groups)

    # Compute attention scores
    attn_weights = torch.matmul(query_states, key_states.transpose(2, 3))
    attn_weights = attn_weights * custom_attention.scaling

    # Apply mask
    if causal_mask is not None:
        attn_weights = attn_weights + causal_mask.to(attn_weights.dtype)

    # Add sinks
    sinks = custom_attention.sinks.view(1, -1, 1, 1).expand(batch_size, -1, seq_len, -1)
    attn_weights_with_sink = torch.cat([attn_weights, sinks.to(attn_weights.dtype)], dim=-1)

    # Softmax
    attn_probs = F.softmax(attn_weights_with_sink, dim=-1, dtype=torch.float32)
    attn_probs_no_sink = attn_probs[..., :-1]

# Convert to numpy for plotting
attn_probs_np = attn_probs_no_sink[0].detach().cpu().float().numpy()  # [num_heads, seq, seq]

# Create visualization
num_heads_to_show = min(8, custom_attention.num_heads)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Attention Patterns Across Different Heads', fontsize=16, fontweight='bold')

for idx in range(num_heads_to_show):
    ax = axes[idx // 4, idx % 4]

    # Plot attention matrix for this head
    im = ax.imshow(attn_probs_np[idx], cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    ax.set_title(f'Head {idx}')

    # Set tick labels to token strings
    token_labels = [t.replace('Ġ', '▁') for t in tokens]
    ax.set_xticks(range(len(token_labels)))
    ax.set_yticks(range(len(token_labels)))
    ax.set_xticklabels(token_labels, rotation=45, ha='right')
    ax.set_yticklabels(token_labels)

    plt.colorbar(im, ax=ax, label='Attention Probability')

plt.tight_layout()
plt.savefig('attention_patterns.png', dpi=150, bbox_inches='tight')
print("Saved visualization to 'attention_patterns.png'")
plt.show()

---

## 11. Analysis: Attention Statistics

In [15]:
print("Attention Statistics Analysis")
print("=" * 60)

# Average attention per head
avg_attn_per_head = attn_probs_np.mean(axis=(1, 2))

print("Average attention probability per head:")
for i, avg in enumerate(avg_attn_per_head[:8]):
    print(f"  Head {i}: {avg:.4f}")

# Attention entropy (measure of concentration)
def compute_entropy(probs):
    """Compute entropy of attention distribution"""
    # Add small epsilon to avoid log(0)
    epsilon = 1e-9
    return -(probs * np.log(probs + epsilon)).sum(axis=-1)

entropy = compute_entropy(attn_probs_np)

print("\nAttention entropy per head (higher = more dispersed):")
for i in range(min(8, len(entropy))):
    print(f"  Head {i}: {entropy[i].mean():.4f}")

# Check causal property
print("\nCausal masking verification:")
for i in range(seq_len):
    for j in range(seq_len):
        if j > i:  # Future positions
            if attn_probs_np[:, i, j].max() > 1e-6:
                print(f"  WARNING: Non-zero attention from pos {i} to future pos {j}")
                break
else:
    print("  ✓ Causal masking is correctly applied!")

print("=" * 60)

Attention Statistics Analysis
Average attention probability per head:
  Head 0: 0.0005
  Head 1: 0.0114
  Head 2: 0.0119
  Head 3: 0.0000
  Head 4: 0.1775
  Head 5: 0.0148
  Head 6: 0.2222
  Head 7: 0.1113

Attention entropy per head (higher = more dispersed):
  Head 0: 0.0099
  Head 1: 0.0939
  Head 2: 0.0916
  Head 3: 0.0000
  Head 4: 0.1076
  Head 5: 0.1310
  Head 6: 0.0221
  Head 7: 0.0036

Causal masking verification:
  ✓ Causal masking is correctly applied!


---

## 12. Detailed Component Analysis

In [16]:
print("Component-wise Analysis")
print("=" * 60)

# Q, K, V statistics
with torch.no_grad():
    q = custom_attention.q_proj(hidden_states)
    k = custom_attention.k_proj(hidden_states)
    v = custom_attention.v_proj(hidden_states)

print("Query statistics:")
print(f"  Shape: {tuple(q.shape)}")
print(f"  Mean: {q.mean().item():.6f}")
print(f"  Std:  {q.std().item():.6f}")
print(f"  Min:  {q.min().item():.6f}")
print(f"  Max:  {q.max().item():.6f}")

print("\nKey statistics:")
print(f"  Shape: {tuple(k.shape)}")
print(f"  Mean: {k.mean().item():.6f}")
print(f"  Std:  {k.std().item():.6f}")

print("\nValue statistics:")
print(f"  Shape: {tuple(v.shape)}")
print(f"  Mean: {v.mean().item():.6f}")
print(f"  Std:  {v.std().item():.6f}")

print("\nSink token values:")
print(f"  {custom_attention.sinks.detach().cpu().float().numpy()[:8]}")

print("\nGQA Key/Value sharing:")
print(f"  {custom_attention.num_kv_heads} KV heads serve {custom_attention.num_heads} Q heads")
print(f"  Sharing ratio: {custom_attention.num_kv_groups}:1")
print(f"  Memory savings: {custom_attention.num_heads / custom_attention.num_kv_heads:.1f}x")

print("=" * 60)

Component-wise Analysis
Query statistics:
  Shape: (1, 3, 4096)
  Mean: 0.008789
  Std:  1.765625
  Min:  -12.125000
  Max:  19.625000

Key statistics:
  Shape: (1, 3, 512)
  Mean: -0.108398
  Std:  12.500000

Value statistics:
  Shape: (1, 3, 512)
  Mean: -0.106445
  Std:  2.453125

Sink token values:
  [ 2.515625    0.55859375  1.71875     0.91015625  0.96875     0.6953125
 -0.98828125  1.421875  ]

GQA Key/Value sharing:
  8 KV heads serve 64 Q heads
  Sharing ratio: 8:1
  Memory savings: 8.0x


---

## 13. Summary and Key Takeaways

Congratulations! You have successfully implemented and verified the Self-Attention mechanism.

### Key Concepts Learned

1. **Self-Attention Mechanism**
   - Query, Key, Value projections
   - Scaled dot-product: prevents gradient issues
   - Softmax normalization: converts scores to probabilities
   - Weighted sum of values: contextual representation

2. **Multi-Head Attention**
   - Multiple parallel attention operations
   - Different heads learn different patterns
   - Final concatenation and projection

3. **Grouped Query Attention (GQA)**
   - Memory-efficient variant of MHA
   - Share K/V across multiple Q heads
   - {custom_attention.num_kv_groups}:1 sharing ratio in GPT-OSS
   - {custom_attention.num_heads / custom_attention.num_kv_heads:.1f}x memory savings

4. **Attention Sinks (GPT-OSS Innovation)**
   - Extra learnable column in attention matrix
   - Provides "opt-out" option when no token is relevant
   - Prevents forced uniform attention distribution
   - Learned per-head parameters

5. **RoPE Integration**
   - Applied to Query and Key (not Value)
   - Encodes relative position information
   - Enables length extrapolation

6. **Causal Masking**
   - Prevents attending to future tokens
   - Essential for autoregressive generation
   - Applied before softmax as additive mask

### Implementation Verification

- ✓ Custom implementation matches HuggingFace exactly
- ✓ Maximum difference: {max_diff:.6e}
- ✓ All attention patterns are causal
- ✓ GQA sharing correctly implemented
- ✓ Sink tokens properly integrated

### Mathematical Review

**Attention Formula:**
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T + \text{Mask}}{\sqrt{d_k}}\right) V
$$

**With Sinks:**
$$
\text{scores} = [QK^T, S]  \quad \text{(concat sink column)}
$$
$$
\text{probs} = \text{softmax}(\text{scores})
$$
$$
\text{output} = \text{probs}[:,:-1] \times V  \quad \text{(remove sink)}
$$

**GQA:**
$$
K_{\text{expanded}} = \text{repeat}(K, n_{\text{groups}})
$$
$$
V_{\text{expanded}} = \text{repeat}(V, n_{\text{groups}})
$$

### References

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - Original Transformer
- [GQA Paper](https://arxiv.org/abs/2305.13245) - Grouped Query Attention
- [Flash Attention](https://arxiv.org/abs/2205.14135) - Efficient attention implementation
- [Sink Tokens Concept] - GPT-OSS specific innovation

---

In [17]:
# Cleanup
print("\nCleaning up...")
import gc

del reference_model
del custom_attention
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared")

print("Tutorial complete!")


Cleaning up...
Tutorial complete!
